# 实战练习：使用 GRPO 微调模型

> **TIP**: 本练习由 LLM 微调专家 [@mlabonne](https://huggingface.co/mlabonne) 撰写。

现在进入实践环节！本练习将完整演示如何使用 GRPO 微调一个模型——从安装依赖到生成推理，完成一个端到端的训练流程。

**本练习使用的资源：**
- 模型：`SmolLM2-135M-Instruct`（135M 参数，适合有限硬件）
- 数据集：`mlabonne/smoltldr`（短篇故事集）
- 任务：训练模型生成接近 50 tokens 长度的摘要
- 硬件：单张 A10G GPU（约 1 小时完成训练，Google Colab 可用）

## 环境安装

In [ ]:
# 安装核心依赖
# datasets: HuggingFace 数据集库
# transformers: 模型加载和推理
# trl: TRL（Transformer Reinforcement Learning）训练库
# peft: LoRA 等参数高效微调方法
# accelerate: 多 GPU/混合精度训练支持
# bitsandbytes: 量化支持（int8/int4）
# wandb: 实验追踪和可视化
!pip install -qqq datasets==3.2.0 transformers==4.47.1 trl==0.14.0 peft==0.14.0 \
    accelerate==1.2.1 bitsandbytes==0.45.2 wandb==0.19.7 --progress-bar off

# flash-attn: Flash Attention 加速（显著提升训练效率，但安装较慢）
!pip install -qqq flash-attn --no-build-isolation --progress-bar off

In [1]:
# 导入所有必要的库
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model       # LoRA 相关
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

## 实验追踪（Weights & Biases）

Weights & Biases（W&B）是强大的实验追踪工具，可以可视化训练曲线（reward、loss、KL 散度等）。

In [2]:
import wandb

# 登录 W&B（需要 API key，可在 wandb.ai 注册获取）
# 也可以跳过 W&B，在 GRPOConfig 中设置 report_to=[] 即可
wandb.login()

# 或者用环境变量方式：
# import os
# os.environ["WANDB_API_KEY"] = "your-api-key"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: goosmanlei (goosmanlei-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 加载数据集

In [3]:
# 加载 smoltldr 数据集
# 该数据集包含短篇故事，适合用于摘要生成任务的 GRPO 训练
dataset = load_dataset("mlabonne/smoltldr")

# 查看数据集结构
print(dataset)
print()

# 查看第一个样本
print("第一个训练样本：")
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
})

第一个训练样本：
{'prompt': "SUBREDDIT: r/tifu\n\nTITLE: TIFU by trying to pet a dog.\n\nPOST: Last night I went to a Hippie May Day Festival/ Camp out. Needless to say, I passed out hard in my tent at the end of the night.Woken by the warmth and light of the morning sun, I emerged from my tent in search of some water to quench my burgeoning thirst. To my delight I spotted a dog scouting the field before me, about 110 meters away. Without delay I dashed towards it, my urge to pet this dog was immeasurable. On the way back to my tent, while running, I just so happened to come upon the most heinous stick I have ever encountered. The bastard was sticking straight out of the earth, cleverly hidden 

## 加载模型

使用 `SmolLM2-135M-Instruct`，一个 135M 参数的小模型：
- 适合有限硬件（单卡 GPU 即可）
- 有 Instruct 版本（已经过指令微调）
- 如果有更强硬件，可尝试 `SmolLM2-1.7B`

In [4]:
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",              # 自动选择精度（通常是 bfloat16）
    device_map="auto",               # 自动分配到可用 GPU/CPU
    attn_implementation="flash_attention_2",  # 使用 Flash Attention 2 加速
)

# 加载对应的 tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(f"模型已加载：{model_id}")
print(f"模型参数量：{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

模型已加载：HuggingFaceTB/SmolLM-135M-Instruct
模型参数量：134.5M


## 配置 LoRA

LoRA（Low-Rank Adaptation）通过在原始模型权重旁边添加低秩矩阵来实现参数高效微调：
- 只训练少量额外参数（通常 < 1% 的原始参数量）
- 大幅降低显存需求
- 训练速度更快

> **TIP**: 如果你不熟悉 LoRA，可以参考 [Chapter 11 第 3 节](https://huggingface.co/learn/course/en/chapter11/3)。

In [5]:
# 配置 LoRA 参数
lora_config = LoraConfig(
    task_type="CAUSAL_LM",       # 因果语言模型任务
    r=16,                         # LoRA 的秩（rank），越大效果越好但参数越多
    lora_alpha=32,                # LoRA 缩放系数（通常设为 r 的 2 倍）
    target_modules="all-linear",  # 对所有线性层应用 LoRA（也可指定特定层）
)

# 将 LoRA 应用到模型
model = get_peft_model(model, lora_config)

# 打印可训练参数信息
model.print_trainable_parameters()
# 输出类似：trainable params: 2,686,976 || all params: 137,701,376 || trainable%: 1.95

trainable params: 4,884,480 || all params: 139,399,488 || trainable%: 3.5039


## 定义奖励函数

本练习使用**长度控制奖励函数**：训练模型生成接近 50 tokens 的回答。

这是一个故意简化的奖励函数，用于演示 GRPO 的学习能力。实际应用中应使用更有意义的任务指标。

In [6]:
# 目标生成长度（token 数量）
ideal_length = 50

def reward_len(completions, **kwargs):
    """
    长度控制奖励函数
    
    奖励值为负，越接近 ideal_length 奖励越高（越接近 0）
    例如：
      - 生成 50 tokens → 奖励 = 0（最高）
      - 生成 30 tokens → 奖励 = -20
      - 生成 100 tokens → 奖励 = -50
    """
    # 注意：这里按 token 数计算长度，而不是字符数
    return [
        -abs(ideal_length - len(tokenizer.encode(completion, add_special_tokens=False)))
        for completion in completions
    ]


# 演示奖励函数行为
examples = [
    "short" * 2,            # 很短
    "medium " * 10,         # 接近目标长度
    "very long " * 20,      # 很长
]

print("奖励函数测试：")
for ex in examples:
    reward = reward_len([ex])[0]
    ex_len = len(tokenizer.encode(ex, add_special_tokens=False))
    print(f"  长度={ex_len:4d} tokens → 奖励={reward:5.1f}")

奖励函数测试：
  长度=   2 tokens → 奖励=-48.0
  长度=  11 tokens → 奖励=-39.0
  长度=  41 tokens → 奖励= -9.0


## 定义训练参数

In [ ]:
# 配置 GRPO 训练参数
# 注：以下参数针对 RTX 5090（32GB）优化，充分利用显存和算力
import inspect

# 先用字典组织参数，便于按版本动态加参数
training_args_kwargs = dict(
    output_dir="GRPO",                    # 输出目录（检查点保存位置）

    # ---- 优化器参数 ----
    learning_rate=2e-5,                   # 学习率
    optim="adamw_8bit",                   # 使用 8-bit AdamW 优化器（节省显存）

    # ---- 批次参数 ----
    # 原始值 8 只用了 1.9GB / 32GB 显存，大幅提升 batch size 来喂饱 GPU
    per_device_train_batch_size=256,       # 从 8 → 256（显存充足时直接扩大）
    gradient_accumulation_steps=1,        # 去掉梯度累积（batch 已足够大）

    # ---- 序列长度参数 ----
    # max_prompt_length 在 trl>=0.15 中已移除，prompt 截断由数据集预处理控制
    max_completion_length=96,             # 生成回答最大长度（token 数）

    # ---- GRPO 核心参数 ----
    # 注意：实际每步处理的 unique prompt 数 = per_device_train_batch_size / num_generations
    #       = 256 / 16 = 16 个 prompt/步
    num_generations=16,                   # 从 8 → 16（显存充足，增加组内多样性）

    # ---- 训练控制 ----
    # 实测 1 epoch（125 步）时 reward 曲线从 -40 升至 -20 仍未收敛，
    # 增加到 4 epochs（~500 步）让模型有充足步数收敛到目标长度 50 附近
    num_train_epochs=4,                   # 1 → 4（约 500 步，预计 reward 收敛至 -5 以内）
    bf16=True,                            # 使用 bfloat16 精度

    # ---- 数据加载（解决 CPU 瓶颈）----
    dataloader_num_workers=4,             # 多进程预加载数据，避免 GPU 等 CPU
    dataloader_pin_memory=True,           # 锁页内存，加速 CPU→GPU 传输

    # ---- 日志和评估 ----
    report_to=["wandb"],                  # 向 W&B 报告指标
    logging_steps=1,                     # 每步记录一次日志

    # ---- 其他 ----
    remove_unused_columns=False,          # 保留数据集中所有列（奖励函数可能需要）
)

training_args = GRPOConfig(**training_args_kwargs)

print("训练参数配置完成")

## 启动训练

In [ ]:
# 初始化 GRPOTrainer
trainer = GRPOTrainer(
    model=model,                          # 已加载的模型（带 LoRA）
    reward_funcs=[reward_len],            # 奖励函数列表（可以传入多个）
    args=training_args,                   # 训练配置
    train_dataset=dataset["train"],       # 训练数据集
)

# 初始化 W&B 实验
wandb.init(project="GRPO")

# 开始训练！
# 4 epochs × 125 步/epoch = 500 步，RTX 5090 上约需 45 分钟
# 实测 1 epoch（125 步）reward 仅从 -40 升至 -20，曲线未收敛；
# 4 epochs 预计 reward 可收敛至 -5 以内（生成长度接近目标 50 tokens）
trainer.train()

## 解读训练结果

训练过程中，`GRPOTrainer` 会记录以下关键指标：

### 奖励（Reward）

奖励值应随训练**逐渐向 0 靠拢**（因为我们的奖励函数最大值是 0）。
这是模型学习到生成接近目标长度的正面信号。

### 损失（Loss）

> **TIP**: GRPO 训练中，**loss 从 0 开始然后上升是完全正常的**，不代表训练出问题！

原因：GRPO 的 loss 正比于 KL 散度（当前策略与参考策略的差异）。随着模型优化以更好地匹配奖励函数，它会自然地偏离初始策略，导致 KL 散度增大，loss 升高。**loss 上升恰恰说明模型在学习**。

核心判断指标：
- ✅ `reward` 持续上升（向目标靠近）
- ✅ `reward_std` 非零（组内有多样性，梯度稳定）
- ⚠️ `kl` 过大（模型偏离参考策略太多，可增大 β）

In [ ]:
# 训练后快速验证：用训练集的样本直接测奖励函数，确认模型是否真正学会了长度约束
# 若 GRPO 有效，reward（取 -abs(50 - len)）应比训练前明显靠近 0

import random
random.seed(42)
val_prompts = [dataset["validation"][i]["prompt"] for i in range(10)]

# 构造 chat messages
def make_messages(prompt):
    return [{"role": "user", "content": prompt}]

gen_kwargs = {"max_new_tokens": 256, "do_sample": True, "temperature": 0.7}

rewards = []
for p in val_prompts:
    out = generator(make_messages(p), **gen_kwargs)
    text = extract_assistant_text(out)
    n = len(tokenizer.encode(text, add_special_tokens=False))
    rewards.append(-abs(50 - n))

avg_reward = sum(rewards) / len(rewards)
print(f"验证集 10 样本平均奖励：{avg_reward:.1f}  （目标：0，初始约 -46）")
print(f"各样本奖励：{rewards}")

## 发布模型到 HuggingFace Hub

In [17]:
# 将 LoRA 权重合并到基础模型（得到完整的模型权重）
merged_model = trainer.model.merge_and_unload()

ft_model_id = "goosmanlei/SmolLM-135M-Instruct-GRPO-smoltldr"

# 推送到 HuggingFace Hub
# private=False：公开模型（设为 True 则为私有）
# tags：添加标签，方便搜索和分类
merged_model.push_to_hub(
    ft_model_id,
    private=False,
    tags=["GRPO", "Reasoning-Course"]
)

# 同时推送 tokenizer
tokenizer.push_to_hub(ft_model_id)

print("模型已成功发布到 HuggingFace Hub！")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

模型已成功发布到 HuggingFace Hub！


## 使用模型进行推理

In [18]:
# 准备测试文档（一篇关于猫的长文章）
prompt = """
# A long document about the Cat

The cat (Felis catus), also referred to as the domestic cat or house cat, is a small 
domesticated carnivorous mammal. It is the only domesticated species of the family Felidae.
Advances in archaeology and genetics have shown that the domestication of the cat occurred
in the Near East around 7500 BC. It is commonly kept as a pet and farm cat, but also ranges
freely as a feral cat avoiding human contact. It is valued by humans for companionship and
its ability to kill vermin. Its retractable claws are adapted to killing small prey species
such as mice and rats. It has a strong, flexible body, quick reflexes, and sharp teeth,
and its night vision and sense of smell are well developed. It is a social species,
but a solitary hunter and a crepuscular predator. Cat communication includes
vocalizations—including meowing, purring, trilling, hissing, growling, and grunting—as
well as body language. It can hear sounds too faint or too high in frequency for human ears,
such as those made by small mammals. It secretes and perceives pheromones.
"""

# 构造 chat 格式的消息
messages = [
    {"role": "user", "content": prompt},  # 用户输入长文档，期望模型生成摘要
]

print("准备推理输入完成")
print(f"文档长度：{len(prompt)} 字符")

准备推理输入完成
文档长度：1082 字符


In [ ]:
from transformers import pipeline

def extract_assistant_text(generated):
    payload = generated[0]["generated_text"]
    if isinstance(payload, list):
        for msg in reversed(payload):
            if isinstance(msg, dict) and msg.get("role") == "assistant":
                return msg.get("content", "")
        return str(payload[-1]) if payload else ""
    return payload

# 方式 1：从 Hub 加载基模
old_generator = pipeline("text-generation", model=model_id)

# 两个模型统一用采样解码（do_sample=True）+ 5 次采样取平均，确保对比条件完全一致
old_generate_kwargs = {
    "max_new_tokens": 256,
    "do_sample": True,
    "temperature": 0.7,
}

n_samples = 5
old_lengths = []
for i in range(n_samples):
    old_gen = old_generator(messages, **old_generate_kwargs)
    old_text = extract_assistant_text(old_gen)
    old_lengths.append(len(tokenizer.encode(old_text, add_special_tokens=False)))
    if i == 0:
        old_first_text = old_text

old_avg_len = sum(old_lengths) / len(old_lengths)

print("=" * 50)
print("基座模型输出（第 1 次采样）：")
print("=" * 50)
print(old_first_text)
print(f"\n各次采样 token 长度：{old_lengths}")
print(f"平均长度：{old_avg_len:.1f}  （目标：50）")

In [ ]:
from transformers import pipeline

# 方式 2：从 Hub 加载已推送的 GRPO 模型
generator = pipeline("text-generation", model=ft_model_id)

# GRPO 训练时使用随机采样（num_generations=16），
# 推理时也应用采样解码（do_sample=True），而非 greedy（do_sample=False）。
# 原因：greedy 每步选最高概率 token，即便模型学会了在 50 tokens 处"倾向于" EOS，
#       EOS 概率若不是绝对最高，greedy 仍会跳过它；
#       采样解码更能反映 GRPO 训练后的概率分布。
generate_kwargs = {
    "max_new_tokens": 256,
    "do_sample": True,
    "temperature": 0.7,
}

# 多次采样取平均，更能反映模型的"期望行为"
n_samples = 5
lengths = []
for i in range(n_samples):
    gen = generator(messages, **generate_kwargs)
    text = extract_assistant_text(gen)
    tok_len = len(tokenizer.encode(text, add_special_tokens=False))
    lengths.append(tok_len)
    if i == 0:
        first_text = text

avg_len = sum(lengths) / len(lengths)

print("=" * 50)
print("GRPO 微调后模型输出（第 1 次采样）：")
print("=" * 50)
print(first_text)
print(f"\n各次采样 token 长度：{lengths}")
print(f"平均长度：{avg_len:.1f}  （目标：50）")
print(f"平均奖励值：{-abs(50 - avg_len):.1f}  （越接近 0 越好）")

## 本节小结

恭喜你完成了第一个 GRPO 微调练习！

### 你学到了什么

1. **完整的 GRPO 训练流程**：从安装依赖到发布模型
2. **LoRA 的使用**：通过参数高效微调减少显存需求
3. **奖励函数设计**：用长度控制函数演示了 GRPO 的学习过程
4. **训练结果解读**：理解了 loss 上升是正常现象，应关注 reward 指标

### 扩展挑战

- 尝试更大的模型（`SmolLM2-1.7B`）
- 设计更复杂的奖励函数（格式 + 内容质量）
- 在数学数据集上训练（如 GSM8K）
- 使用 `push_to_hub=True` 让训练过程中定期自动推送检查点

---

**下一节**：使用 Unsloth 加速 GRPO 训练，在免费 T4 GPU 上也能运行！